In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sb
from matplotlib.ticker import (MultipleLocator)

rng = np.random.default_rng()

%matplotlib inline

In [ ]:
save_bool=True #A boolean that controls if figures and tables are saved. (True=save)

### 1. Read the data

In [ ]:
raw_data = pd.read_excel('data/RawData.xlsx') #read the raw data

### 2. Convert to dry weight

In [ ]:
#correct the std for cases where the std is derived from the full reported range, as half this range
indx = ((raw_data['standard deviation']) == (raw_data['numerical value'])) & ((raw_data['standard deviation']) >0)
raw_data.loc[indx , 'standard deviation'] /= 2

#Correct units for dry weight using a general factor of 0.3
raw_data.loc[:,'norm value'] = raw_data.loc[:,'numerical value']
raw_data.loc[raw_data.units=='mg/m^2 (wet weight)','norm value'] *= 0.3  #convert the wet mass to an effective dry mass, by multiplying by 0.3

raw_data.loc[raw_data.units=='mg/m^2 (wet weight)','standard deviation'] *= 0.3 #convert the wet mass to an effective dry mass, by multiplying by 0.3 - for the std 
raw_data.loc[raw_data['standard deviation'].isna(),'standard deviation'] = 0

# fine-tune taxa-specific conversion rates from wet to dry mass (from Petersen & Luxton)
conversion_rates = {'taxon':['Isoptera', 'Acari', 'Collembola', 'Formicidae','Araneae','Pseudoscorpiones'],
                     'rank':['super-family', 'sub-class','sub-class', 'family','order','order'],
                      'ratio':[0.27, 0.4,0.32,0.228,0.43,0.224]}
 
convR = pd.DataFrame(conversion_rates)

for ii in convR.index:
    case = pd.concat([raw_data[convR.loc[ii,'rank']]==convR.loc[ii,'taxon'], raw_data.units=='mg/m^2 (wet weight)'], axis=1).all(axis=1)
    raw_data.loc[case,'norm value']= raw_data.loc[case,'numerical value']*convR.loc[ii,'ratio'] 
    raw_data.loc[case,'standard deviation'] *= convR.loc[ii,'ratio'] /0.3


#units type distinguishes population to biomass measurements
def unit_type(x):
    X=x.partition('/')[0]
    return X

raw_data.loc[:,'units type'] = raw_data.units.apply(unit_type) #units type distinguishes population to biomass measurements

raw_data.groupby('units type').site.nunique()#display the counts for the separated data

### 3. Remove partial measurements, and those with unknown biomes

In [ ]:
valid_data = raw_data.copy()
valid_data['studied group'] = valid_data['studied group'].str.strip()
valid_data = valid_data[(valid_data['studied group']!='Microarthropods') | ((valid_data['studied group']=='Microarthropods') & (valid_data['sub-class'].isin(['Acari','Collembola'])) & (valid_data['aggregated environment'] =='soil/litter') )]

#Drop ants measurements with unknown biomes
valid_data = valid_data[valid_data['aggregated biome'] != 'Shrubland/Grassland']

### 4. Classify into taxonomic and habitat type groups

In [ ]:
valid_data.loc[valid_data['sub-class']=='Acari','aggregated taxon'] = 'Acari'
valid_data.loc[valid_data['sub-class']=='Collembola','aggregated taxon'] = 'Collembola'
valid_data.loc[valid_data['super-family']=='Isoptera','aggregated taxon'] = 'Isoptera'
valid_data.loc[valid_data['family']=='Formicidae','aggregated taxon'] = 'Formicidae'
valid_data.loc[valid_data['aggregated taxon'].isna(),'aggregated taxon'] = 'Other'

valid_data.loc[valid_data['aggregated environment'] !='soil/litter','aggregated taxon'] = 'Combined'

In [ ]:
soil_data = valid_data[valid_data['aggregated environment'] =='soil/litter']
above_ground_data = valid_data[valid_data['aggregated environment'] !='soil/litter']

### 5. Sum and average over taxa per site, and calculate the effective mass of an individual soil arthropod

In [ ]:
#Average the soil data over each taxon in each site, then sum all taxons in each site according to the aggregated groups and data type they are in.
soil_site_taxa_mean = soil_data.groupby(['units type','aggregated taxon','aggregated biome','site','taxon'])['norm value'].mean().reset_index()
soil_site_data = soil_site_taxa_mean.groupby(['units type','aggregated taxon','aggregated biome','site'])['norm value'].sum().reset_index()

#Divide into the two types of measurements
soil_site_data_pop = soil_site_data[soil_site_data['units type']=='individuals']
soil_site_data_pop.rename(columns={'norm value': 'population density'}, inplace=True)

soil_site_data_mass = soil_site_data[soil_site_data['units type']=='mg']
soil_site_data_mass.rename(columns={'norm value': 'mass density'}, inplace=True)

#Construct a new dataframe, where we keep only the sites with both measurement types
soil_site_data_comb = pd.merge( soil_site_data_pop, soil_site_data_mass, on=["aggregated taxon","aggregated biome","site"], how="inner", validate="one_to_one" )

#calculate the mass of an individual per site, in units of mg/ind
soil_site_data_comb.loc[:,'ind mass'] = soil_site_data_comb.loc[:,'mass density']/soil_site_data_comb.loc[:,'population density']

#Present the mass of an individual per biome and taxon, in units of mg/ind
##soil_site_data_comb.pivot_table(index='aggregated taxon',columns='aggregated biome',values='ind mass', aggfunc=['mean','count'])

print('Number of sites with combined data: ' + '%d' %soil_site_data_comb.site.nunique())

In [ ]:
#filter out outliers for the effective mass of an individuals
def filter_outliers(x):
    if len(x)>3:
        STD=x.std()
        Mean=np.mean(x)
        x = x[x<(Mean+2*STD)]
        x = x[x>(Mean-2*STD)]
    return x

soil_ratio_filt = soil_site_data_comb.groupby(['aggregated taxon','aggregated biome'])['ind mass'].apply(filter_outliers)
soil_ratio_filt_full = soil_ratio_filt.reset_index().pivot_table(index='aggregated taxon',columns='aggregated biome',values='ind mass', aggfunc='mean')
soil_ratio_filt_total = soil_ratio_filt.reset_index().groupby('aggregated taxon')['ind mass'].mean() #the global mean for the mass ratio per taxon
soil_ratio_filt_full = pd.merge(soil_ratio_filt_full, soil_ratio_filt_total,on="aggregated taxon", how="inner", validate="one_to_one" )
soil_ratio_filt_full.rename(columns={'ind mass': 'Total average (mg/ind)'}, inplace=True)

#fill in the nans with the global averages and save to csv
for clm in soil_ratio_filt_full.columns:
    soil_ratio_filt_full.loc[soil_ratio_filt_full[clm].isnull(),clm] = soil_ratio_filt_full['Total average (mg/ind)']

biomes = soil_site_data_pop.loc[:,'aggregated biome'].unique()

for biom in biomes: 
    if biom not in soil_ratio_filt_full.columns:
        soil_ratio_filt_full.loc[:,biom] = soil_ratio_filt_full.loc[:,'Total average (mg/ind)']

if save_bool:        
    soil_ratio_filt_full.to_csv('results/tables/average_ind_mass_full.csv')      

def print_ratio(x):
    return '{:.2e}'.format(x)

soil_ratio_filt_print = soil_ratio_filt_full.applymap(print_ratio) #with rounded values

### 6. Define all functions for the bootstrapping process

In [ ]:
#General default parameters (see uses below):
N_boot=100000 #number of resamples used for bootstrapping (can be lowered to reduce execution time)
N_samps = 100000 #number of samples used for each taxon-biome CDF (can be lowered to reduce execution time)
Nbins=500 #The number of histogram bins used to convert the means array into a descrete cdf - cumulative distribution function

col_arr = ['#2240FF','#00BF9F','#FFBB00','#f44336','#9e59b3'] #color array for soil taxa in the order: 'Acari','Collembola','Formicidae','Isoptera','Other'


#Bootstrap using random choose with replacement. returns an array of means
def Boot_means(vals, sigma,Boot_N):
    L = len(vals)
    means = np.zeros(Boot_N) 

    for ii in range(Boot_N): #random choose with replacement
        indx = rng.choice(np.array(range(L)),L, replace=True)

        Boot = np.zeros(L)
        jj=0
        for ind in indx: #add the measurement uncertainties using a normal distribution
            Boot[jj] = np.max( [ 0 , rng.normal(vals[ind], sigma[ind], 1) ])
            jj +=1
        
        means[ii] = np.mean(Boot) #take the mean of a single resample
    
    return means

In [ ]:
#Get the aggregated biome areas
biome_area = pd.read_csv('data/aggregated biomes data.csv') # biomes area in units of m^2
biome_area1 = biome_area.groupby('aggregated biome 1')['area'].sum().reset_index().rename(columns={'aggregated biome 1': 'aggregated biome'}) #aggregate the biomes areas

def sum_error(x):
    return np.sqrt(np.sum(x**2))

def mean_error(x):
    return sum_error(x)/np.sqrt(len(x))

#Average the soil data over each taxon in each site, then sum all taxons in each site according to the aggregated groups and data type they are in.
def averaged_sum_site(data,GroupBy=['aggregated taxon','aggregated biome']):
    GroupBy = GroupBy + ['site','taxon']

    data_taxa_mean = data.groupby(GroupBy)['norm value'].mean().reset_index() # mean per taxon and site in each group
    data_site = data_taxa_mean.groupby(GroupBy[:-1])['norm value'].sum().reset_index() # sum over all taxa in each site for each group
    
    data_taxa_mean_std = data.groupby(GroupBy)['standard deviation'].apply(mean_error).reset_index() # mean per taxon and site in each group
    data_site_std = data_taxa_mean_std.groupby(GroupBy[:-1])['standard deviation'].apply(sum_error).reset_index() # sum over all taxa in each site for each group

    return data_site.merge(data_site_std)

#get bootstrapped means using pandas
def boot_means2(x,N): 
    means=np.zeros(N)
    for ii in range(N):
        smp = x.sample(frac=1, replace=True, weights=None, random_state=None, axis=None).iloc[:,[-2,-1]]
        rnds = rng.normal(smp.iloc[:,0].values,smp.iloc[:,1].values)
        rnds[rnds<0]=0
        means[ii]=np.mean(rnds)

    return means

#Extract mean and 95% CI from bootstrapped data (returns an array)
def Boot_stats(means, prcnt_low=2.5, prcnt_high=97.5):    
    Mean = np.mean(means)
    prcnt = np.percentile(means,[prcnt_low,prcnt_high])
    return np.array([Mean , prcnt[0], prcnt[1]])


#Functions that help to handel the bootstrapped data and help in the Monte Carlo process for the total sum
def means_to_cdf(means, Nbins=Nbins): #converts the means array into a descrete cdf (cumulative distribution function) using a histogram with Nbins bins. 
    hist, bin_edges = np.histogram(means, bins = Nbins, density = False)
    bin_centers = (bin_edges[:-1] + bin_edges[1:] )/2
    cdf = np.cumsum(hist)
    cdf = cdf / cdf[-1]
    return cdf, bin_centers

#bootstrap and return compressed results using pandas
def boot(data,GroupBy=['aggregated taxon','aggregated biome'],N=N_boot): 
    means = data.groupby(GroupBy).apply(lambda x: pd.Series(boot_means2(x,N),index=range(N))) #generate means array for each group
    Stats = means.apply(lambda x: pd.Series(Boot_stats(x),index=['mean','2.5%','97.5%']) ,axis=1) #extract stats from these arrays
    CDF = means.apply(lambda x: pd.Series(means_to_cdf(x),index=['cdf','values']),axis=1) #Extract a cumulative distribution function for Monte-Carlo etc.
    return Stats, CDF

def random_from_cdf(cdf,bin_centers,N): #picks N values of bin_centers according to the histogram's cdf
    values = np.random.rand(N)
    value_bins_indx = np.searchsorted(cdf, values)
    rnd_from_cdf = bin_centers[value_bins_indx]
    return rnd_from_cdf

#Get global (biome-level) stats from the density stats
def dens_2_tot(Stats,area=biome_area1):
    biome_means = Stats.reset_index().merge(area)
    biome_means.loc[:,'Total'] = (biome_means.loc[:,'mean']*biome_means.loc[:,'area']).values/1e15 #total biomass per biome, in units of Mt
    biome_means.loc[:,'Total_low'] = (biome_means.loc[:,'2.5%']*biome_means.loc[:,'area']).values/1e15 #total low biomass per biome, in units of Mt
    biome_means.loc[:,'Total_high'] = (biome_means.loc[:,'97.5%']*biome_means.loc[:,'area']).values/1e15 #total high biomass per biome, in units of Mt
    return biome_means

#Convert CDF to absolute units (from density to absolute in each biome), and sum using a Monte-Carlo process
def cdf_2_tot(CDF,area=biome_area1, N_samps = N_samps): #area is a pd.DataFrame, N_samps = number of samples used for each taxon-biome CDF

    area = pd.DataFrame(area).set_index('aggregated biome')
    dist_tot = CDF.merge(area,left_on='aggregated biome',right_index=True) #add the total areas of the aggregated biomes 
    dist_tot.loc[:,'values'] = (dist_tot['values']*dist_tot['area'])
    dist_tot.drop('area',axis='columns',inplace=True)
    samps_dist_tot = dist_tot.apply(lambda x: pd.Series(random_from_cdf(x['cdf'],x['values'],N_samps),index=range(N_samps)),axis=1)
    return samps_dist_tot/1e15

#combine total biomass means of several habitats (called x,y,z)
def comb_tots(x,y,N_samps=N_samps,Nbins=Nbins):
    combined = pd.DataFrame(columns=['cdf','tots'])
    ii=0
    for habitat in [x,y]:
        cdf, tots = means_to_cdf(habitat, Nbins)
        combined.loc[ii,:]=[[cdf], [tots]]
        ii+=1
    combined = combined.apply(lambda x: pd.Series(random_from_cdf(x['cdf'][0],x['tots'][0],N_samps),index=range(N_samps)),axis=1)
    return combined.sum()
    

In [ ]:
#Split data according to unit type (mass or popiulation) and use mass of individuals to convert all to mass (as a function)
def average_with_eff_mass(data,GroupBy=['units type','aggregated taxon','aggregated biome']):
    
    x = averaged_sum_site(data,GroupBy)

    #Divide into the two types of measurements
    x_pop = x[x['units type']=='individuals']
    x_pop.rename(columns={'norm value': 'population density','standard deviation': 'pop_std'}, inplace=True)

    x_mass = x[x['units type']=='mg']
    x_mass.rename(columns={'norm value': 'mass density','standard deviation': 'mass_std'}, inplace=True)

    #Construct a new dataframe, where we keep only the sites with both measurement types
    x_comb = pd.merge( x_pop, x_mass, on=GroupBy[1:]+['site'], how="inner", validate="one_to_one" )

    #find the measurements where only population data is available
    cond1 = x_pop['site'].isin(x_comb['site']) #intersection of cond1 and cond2 will be excluded
    cond2 = x_pop['population density'].isin(x_comb['population density']) 
    x_pop_pure = x_pop.drop(x_pop[cond1 & cond2].index )#measurements with only population measurements, and no mass measurements, based on site name and numerical value    
    
    #read the weight of an individual taxon in each biome, and also globaly
    soil_ratio_filt_full = pd.read_csv('results/tables/average_ind_mass_full.csv').set_index('aggregated taxon')         
    
    
    #convert population to effective mass based on biome level (_B) or global level (_G)
    def to_mass_B(x_pop_pure,ind_mass = soil_ratio_filt_full):
        if x_pop_pure.loc['aggregated taxon'] != 'Other': #We don't calculate an effective mass for 'Others', as this group is too diverse
            return x_pop_pure.loc[['population density','pop_std']]* ind_mass.loc[x_pop_pure.loc['aggregated taxon'],x_pop_pure.loc['aggregated biome']]
        else:
            return np.nan

    def to_mass_G(x_pop_pure,ind_mass = soil_ratio_filt_full):
        if x_pop_pure.loc['aggregated taxon'] != 'Other': #We don't calculate an effective mass for 'Others', as this group is too diverse
            return x_pop_pure.loc[['population density','pop_std']]* ind_mass.loc[x_pop_pure.loc['aggregated taxon'],'Total average (mg/ind)']
        else:
            return np.nan

        
    X = x_pop_pure.apply(to_mass_B,axis=1)
    X.rename(columns={'population density':'eff_mass_B','pop_std':'eff_mass_std_B'}, inplace=True)
    x_pop_pure = x_pop_pure.join(X)
    
    X = x_pop_pure.apply(to_mass_G,axis=1)
    X.rename(columns={'population density':'eff_mass_G','pop_std':'eff_mass_std_G'}, inplace=True)
    x_pop_pure = x_pop_pure.join(X)
        
    #Merge all the measurements, converted into mass
    x_mass_all = pd.merge(x_mass,x_pop_pure,on=GroupBy[1:]+['site'], how="outer") 

    x_mass_all.loc[:,'mass_B'] = x_mass_all.loc[:,['mass density','eff_mass_B']].apply(np.nansum,axis=1)
    x_mass_all.loc[:,'mass_B_std'] = x_mass_all.loc[:,['mass_std','eff_mass_std_B']].apply(np.nansum,axis=1)
    x_mass_all.loc[:,'mass_G'] = x_mass_all.loc[:,['mass density','eff_mass_G']].apply(np.nansum,axis=1)
    x_mass_all.loc[:,'mass_G_std'] = x_mass_all.loc[:,['mass_std','eff_mass_std_G']].apply(np.nansum,axis=1)

    x_mass_all = x_mass_all[~np.isnan(x_mass_all.loc[:,'mass_G'])] # remove measurements of "other" taxa where there is no direct biomass measurement

    return x_mass_all


In [ ]:
#printing functions:
def prtF(x):#print format
    return '{:.0f}'.format(x)

def prnt_monte(Overall_tots):
    return 'Median: ' + prtF(Overall_tots.median()) +' (95% CI:'+ prtF(np.percentile(Overall_tots,[2.5])[0]) +'-'+ prtF(np.percentile(Overall_tots,[97.5])[0]) +') Mt'

def prnt_stats(tots):
    return 'Mean: ' + prtF(tots.Total.sum()) +' (Range:'+ prtF(tots.Total_low.sum()) +'-'+ prtF(tots.Total_high.sum()) +') Mt'

#for population data:
def prtF_pop(x):#print format 
    return '{:.1e}'.format(x)

def prnt_monte_pop(Overall_tots):
    return 'Median: ' + prtF_pop(Overall_tots.median()) +' (95% CI:'+ prtF_pop(np.percentile(Overall_tots,[2.5])[0]) +'-'+ prtF_pop(np.percentile(Overall_tots,[97.5])[0]) +') individuals'

def prnt_stats_pop(tots):
    return 'Mean: ' + prtF_pop(tots.Total.sum()) +' (Range:'+ prtF_pop(tots.Total_low.sum()) +'-'+ prtF_pop(tots.Total_high.sum()) +') individuals'

In [ ]:
def plot_biomass(Overall_tots, color='b', xlabel = 'Total Dry Biomass [Mt]',
                 ylabel='Probability Density\n[arb. units]', save_bool = False, filename='test',title='',norm_to_1=True, show=True):
    plt.rc('xtick', labelsize=14) 
    plt.rc('ytick', labelsize=14)

    plt.hist(Overall_tots,bins=200,alpha=1,color=color)
    
    if norm_to_1:
        ax = plt.gca()
        mx= ax.get_ylim()[1]*0.95 
        ax.yaxis.set_ticks([0, mx/2, mx])
        ax.yaxis.set_ticklabels([0, 0.5, 1])
    
    plt.xlabel(xlabel,size = 18)
    plt.ylabel(ylabel,size = 18)
    plt.title(title,size = 20)

    xc = np.percentile(Overall_tots,[2.5,50,97.5])
    plt.axvline(x=xc[0], color='k', linestyle='--',alpha=0.5, label = '2.5%')
    plt.axvline(x=xc[1], color='r', linestyle='-',alpha=0.7, label = 'Median')
    plt.axvline(x=xc[2], color='k', linestyle='--',alpha=0.5, label = '97.5%')
    
    plt.tight_layout()
    plt.legend(loc='upper right')
    
    if save_bool:        
        plt.savefig('results/figs/' + filename + '.pdf',dpi=300)
    else:
        if show:
            plt.show()
        
Habitat_Color=['#945200','#FFD479','#008F00'] 
col_env = [Habitat_Color[0],Habitat_Color[2]]
col_soil = Habitat_Color[0]

### 7. Calculate biomasses using bootstrapping

In [ ]:
#Soil: 
soil_site_data_mass_all = average_with_eff_mass(soil_data,GroupBy=['units type','aggregated taxon','aggregated biome']) #data to be used. Contains all soil data, converted into dry biomass
soil_Stats, CDF = boot(soil_site_data_mass_all,GroupBy=['aggregated taxon','aggregated biome'],N=N_boot)
soil_tots = dens_2_tot(soil_Stats,area=biome_area1)
print('Total soil biomass from Stats (extended range): \n' + prnt_stats(soil_tots))

samp_dist_tots = cdf_2_tot(CDF,area=biome_area1, N_samps = N_samps)
soil_Overall_tots = samp_dist_tots.sum(axis=0)
print("Total soil biomass from Monte-Carlo: \n"+ prnt_monte(soil_Overall_tots))

plot_biomass(soil_Overall_tots,color=Habitat_Color[0], xlabel = 'Dry Biomass in Soil [Mt]', ylabel='Probability Density\n[arb. units]',
             save_bool = save_bool , filename='Soil Biomass')

tot_biomass_taxon = samp_dist_tots.groupby('aggregated taxon').sum().T

In [ ]:
if save_bool:
    soil_site_data_mass_all.to_csv('results/processed_site_data_mean_std.csv')

In [ ]:
tmp = soil_tots.groupby('aggregated taxon')[['Total','Total_low','Total_high']].sum().applymap(prtF)

if save_bool:
    tmp.to_csv('results/tables/totals_per_soil_taxa.csv')
else:
    print(tmp)

In [ ]:
if save_bool:
    soil_site_data_mass_all.to_excel('soil_site_data_mass_all.xlsx')

In [ ]:
def round_significant_N(x,N): #round the number to N significant digits
    if x!=0:
        x_pos = np.abs(x)
        digits = int(np.floor(np.log10(x_pos))+1) #number of digits before the decimal point in x
        decimals = -(digits-N) #number of decimals in the final result
        X = np.round(x, decimals)

        if decimals<1: #convert to an integer if possible
            X=int(X)
    else: 
        X=int(0)
    return X

def print_export(mean,low,high,N_sites,N=2): #print the mean and range (low-high) rounded to N significant digits, for exporting data tables
    mean = round_significant_N(mean,N)
    low = round_significant_N(low,N)
    high = round_significant_N(high,N)
    
    txt='{MEAN} ({LOW}-{HIGH})\nN = {N_SITES}'
    return txt.format(MEAN=mean, LOW=low, HIGH=high, N_SITES=int(N_sites))

In [ ]:
#export soil biomass density table
soil_export = soil_Stats.reset_index().merge(soil_site_data_mass_all.groupby(['aggregated taxon','aggregated biome']).site.count().reset_index())
soil_export = soil_export.groupby(['aggregated taxon','aggregated biome']).mean()
soil_export = soil_export.apply(lambda x : print_export(x['mean'], x['2.5%'],x['97.5%'],x['site'],N=1),axis=1).reset_index()
soil_export.rename(columns={'aggregated taxon' : 'Aggregated Taxon', 'aggregated biome': 'Aggregated Biome',0: 'Density (95% Range)', 'site': '# Sites'}, inplace=True)
soil_density_table = pd.pivot(soil_export, values=['Density (95% Range)'], index='Aggregated Biome',
                    columns='Aggregated Taxon')

if save_bool:
    soil_density_table.to_csv('results/tables/Soil density table.csv')

In [ ]:
#Above ground:
above_ground_data_mass = above_ground_data[above_ground_data['units type']=='mg']

above_ground_data_site = averaged_sum_site(above_ground_data_mass,GroupBy=['aggregated biome'])
above_ground_Stats, CDF = boot(above_ground_data_site,GroupBy=['aggregated biome'],N=N_boot)
above_ground_tots = dens_2_tot(above_ground_Stats,area=biome_area1)
print('Total above ground biomass from Stats (extended range): \n' + prnt_stats(above_ground_tots))

samp_dist_tots = cdf_2_tot(CDF,area=biome_area1, N_samps = N_samps)
above_ground_Overall_tots = samp_dist_tots.sum(axis=0)
print("Total above ground biomass from Monte-Carlo: \n"+ prnt_monte(above_ground_Overall_tots))

plot_biomass(above_ground_Overall_tots,color=Habitat_Color[2], xlabel = 'Above-Ground Dry Biomass [Mt]', ylabel='Probability Density\n[arb. units]',
             save_bool = save_bool , filename='Above-Ground Biomass')

In [ ]:
#Total biomass:
Overall_biomass = comb_tots(soil_Overall_tots,above_ground_Overall_tots,N_samps=N_samps,Nbins=Nbins)
print('Overall_biomass\n'+ prnt_monte(Overall_biomass))

Overall_low = soil_tots.Total_low.sum()+above_ground_tots.Total_low.sum()
Overall_high = soil_tots.Total_high.sum()+above_ground_tots.Total_high.sum()
print('Range: ' + prtF(Overall_low) + ' - ' + prtF(Overall_high) + ' Mt')

plot_biomass(Overall_biomass,color='#1e2fc9', xlabel = 'Dry Biomass [Mt]', ylabel='Probability Density\n[arb. units]',
             save_bool = save_bool , filename='All Biomass')

### 8. Prepare density plot 

In [ ]:
def legend_without_duplicate_labels(ax,pos):
    handles, labels = ax.get_legend_handles_labels()
    mapping = [ ['Acari', 'Acari (Mites)    '],['Collembola', 'Collembola (Springtails)    '],['Formicidae','Formicidae (Ants)    '],['Isoptera', 'Isoptera (Termites)    '],['Other','Others']]
    labels2 = [x.replace(subM[0], subM[1]) for x in labels for subM in mapping if subM[0] in x]
    unique = [(h, l) for i, (h, l) in enumerate(zip(handles, labels2)) if l not in labels2[:i]]
    ax.legend(*zip(*unique),bbox_to_anchor=pos,loc = 'upper center', ncol=5,frameon=False)


In [ ]:
plt.rcParams.update({'font.size': 10})
plt.rc('text', usetex=False)

#r is the data to be plotted
r = soil_site_data_mass_all.reset_index().rename(columns={'mass_G':'numerical value'}) #'numerical value' is the column being used later

#Sort according to biomes total mean biomass, in a descending order
sorter = soil_Stats.reset_index().groupby('aggregated biome').sum().sort_values(by='mean',ascending=False).index
sorterIndex = dict(zip(sorter, range(len(sorter))))
r['sorting_biomes'] = r['aggregated biome'].map(sorterIndex)
r.sort_values(['sorting_biomes'],ascending = True, inplace = True)
r.drop('sorting_biomes', axis='columns', inplace = True)

original_biomes = r['aggregated biome'].unique()
transdict_caps = {'Boreal Forests/Taiga': 'Boreal\nForests/\nTaiga\n(16M km$^2$)',
             'Crops':'Crops\n(15M km$^2$)',
             'Deserts and Xeric Shrublands':'Deserts\nand Xeric\nShrublands\n(20M km$^2$)',
             'Mediterranean Forests, Woodlands and Scrub':'Mediterranean\nForests,\nWoodlands\nand Scrub\n(1.6M km$^2$)',
             'Pasture':'Pasture\n(28M km$^2$)',
           'Temperate Forests':'Temperate\nForests\n(11M km$^2$)',
           'Temperate Grasslands, Savannas and Shrublands':'Temperate\nGrasslands,\nSavannas and\nShrublands\n(5.4M km$^2$)',
           'Tropical and Subtropical Forests':'Tropical and\nSubtropical\nForests\n(18M km$^2$)',
           'Tropical and Subtropical Grasslands, Savannas and Shrublands':'Tropical and\nSubtropical\nGrasslands,\nSavannas and\nShrublands\n(10M km$^2$)',
           'Tundra':'Tundra\n(11M km$^2$)'
            }

transdict = {'Boreal Forests/Taiga': 'Boreal\nforests/\ntaiga\n(16M km$^2$)',
             'Crops':'Crops\n(15M km$^2$)',
             'Deserts and Xeric Shrublands':'Deserts\nand xeric\nshrublands\n(20M km$^2$)',
             'Mediterranean Forests, Woodlands and Scrub':'Mediterranean\nforests,\nwoodlands\nand scrub\n(1.6M km$^2$)',
             'Pasture':'Pasture\n(28M km$^2$)',
           'Temperate Forests':'Temperate\nforests\n(11M km$^2$)',
           'Temperate Grasslands, Savannas and Shrublands':'Temperate\ngrasslands,\nsavannas and\nshrublands\n(5.4M km$^2$)',
           'Tropical and Subtropical Forests':'Tropical and\nsubtropical\nforests\n(18M km$^2$)',
           'Tropical and Subtropical Grasslands, Savannas and Shrublands':'Tropical and\nsubtropical\ngrasslands,\nsavannas and\nshrublands\n(10M km$^2$)',
           'Tundra':'Tundra\n(11M km$^2$)'
            }
print_biomes = [transdict[biome] for biome in original_biomes]
r.replace(to_replace=original_biomes,value=print_biomes,inplace=True)

bins = np.logspace(-1,np.log10(r['numerical value'].max()),20)
bins = np.insert(bins,0,-0.1) #add the point -0.1 to the beginning of the bins array
  
t = r.groupby(['aggregated biome','aggregated taxon'])['numerical value'].apply(lambda x: pd.cut(x,bins=bins).value_counts()/len(x)).reset_index() #how many in each bin
t_max = t.groupby(['aggregated biome','aggregated taxon'])['numerical value'].apply(np.max).reset_index() 
t_norm = t.merge(t_max,left_on=['aggregated biome','aggregated taxon'],right_on=['aggregated biome','aggregated taxon']) #normalize to the maximal bin value
t_norm['val'] = t_norm['numerical value_x']/t_norm['numerical value_y']

In [ ]:
#calculate data normalized according to area
mg_m2 = (tot_biomass_taxon.median()*1e15) / biome_area1.sum()[1]
mg_ind = soil_ratio_filt_full.loc[:,'Total average (mg/ind)']
num_m2 = mg_m2 / mg_ind

print('area:' + '{:.2f}'.format(biome_area1.sum()[1]/1e14) + ' x 1e14 m^2')
print('\nBiomass density (mg/m^2) - in soil: (' +  '{:.0f}'.format(mg_m2.sum()) + ' total )')
print(mg_m2)
print('\nPopulation density from biomass density (ind./m^2) - in soil: (' +  '{:.0f}'.format(num_m2.sum()) + ' total )')
print(num_m2)

print('\nTotal population density from 10^19 individuals:' +  '{:.0f}'.format(1e19/biome_area1.sum()[1]))

### Calculate the soil biomass under different assumptions (using sub-phyla etc.), without using abundance

In [ ]:
#calculate the biomass completely, allowing to change grouping assumptions and data
def calc_boot_full(data,GroupBy,filename='test',N_boot=10000,N_samp=10000,Nbins=500,return_dist=False,
                   is_population=False,save_bool=save_bool,xlabel = 'Dry Biomass [Mt]',ylabel='Probability Density\n[arb. units]',color=Habitat_Color[0]):

    data_site = averaged_sum_site(data,GroupBy)
    Stats, CDF = boot(data_site,GroupBy,N=N_boot)
    tots = dens_2_tot(Stats,area=biome_area1)
    if is_population:
        tots.loc[:,['Total','Total_low','Total_high']]*=1e15
        print('Total Population from Stats (extended range): \n' + prnt_stats_pop(tots))
        samp_dist_tots = cdf_2_tot(CDF,area=biome_area1, N_samps = N_samps)
        samp_dist_tots*=1e15
        Overall_tots = samp_dist_tots.sum(axis=0) *1e-18
        print("Total population from Monte-Carlo: \n"+ prnt_monte_pop(Overall_tots*1e18))
    
    else:
        print('Total biomass from Stats (extended range): \n' + prnt_stats(tots))
        samp_dist_tots = cdf_2_tot(CDF,area=biome_area1, N_samps = N_samps)
        Overall_tots = samp_dist_tots.sum(axis=0)
        print("Total biomass from Monte-Carlo: \n"+ prnt_monte(Overall_tots))
    

    plot_biomass(Overall_tots,color=color, xlabel = xlabel, ylabel=ylabel,
                 save_bool = save_bool , filename=filename,norm_to_1=True, show=False)
    if return_dist:
        return Stats, tots, Overall_tots, samp_dist_tots
    else:
        return Stats, tots, Overall_tots

In [ ]:

soil_data_mass = soil_data[soil_data['units type']=='mg'] #Soil data with only direct mass measurements

#aggregating soil data by: sub-phyla, biome
sp1_Stats,sp1_tots, sp1_Overall_tots = calc_boot_full(data=soil_data_mass,GroupBy=['sub-phylum','aggregated biome'],filename='subphyla soil Biomass',
               N_boot=N_boot,N_samp=N_samps,Nbins=Nbins,
               save_bool=save_bool,xlabel = 'Dry Biomass in Soil [Mt]')


In [ ]:
# Sensitivity analysis for soil data: check how using only mass data would change the results    
fig,ax = plt.subplots(1,2,figsize=[12,4],dpi=300)
ax = plt.subplot(1,2,1)

#aggregating mass soil data by: aggregated taxon, aggregated biome
print('Aggregating mass soil data by: taxon, biome')
b_Stats,b_tots, b_Overall_tots = calc_boot_full(data=soil_data_mass,GroupBy=['aggregated taxon','aggregated biome'],filename='mass only soil Biomass',
               N_boot=N_boot,N_samp=N_samps,Nbins=Nbins,
               save_bool=False,xlabel = 'Dry Biomass in Soil [Mt]')

ax.text(0.01,0.9,'A',fontdict={'size':20},transform = ax.transAxes)

ax = plt.subplot(1,2,2)

#aggregating soil data by: aggregated taxon, sub-phyla, aggregated biome
print('\nAggregating soil data by: taxon, sub-phyla, biome')
tx_sp_Stats,tx_sp_tots, tx_sp_Overall_tots = calc_boot_full(data=soil_data_mass,GroupBy=['aggregated taxon','sub-phylum','aggregated biome'],filename='taxa and subphyla soil Biomass',
               N_boot=N_boot,N_samp=N_samps,Nbins=Nbins,
               save_bool=False,xlabel = 'Dry Biomass in Soil [Mt]',ylabel='')

ax.text(0.01,0.9,'B',fontdict={'size':20},transform = ax.transAxes)

plt.tight_layout()

In [ ]:
#calculate soil biomass by also dividing according to sub-phyla, including all possible datapoints, using effective mass:
soil_site_data_mass_all_SP = average_with_eff_mass(soil_data,GroupBy=['units type','aggregated taxon','sub-phylum','aggregated biome']) #data to be used. Contains all soil data, converted into dry biomass
soil_Stats_SP, CDF_SP = boot(soil_site_data_mass_all_SP,GroupBy=['aggregated taxon','sub-phylum','aggregated biome'],N=N_boot)
soil_tots_SP = dens_2_tot(soil_Stats_SP,area=biome_area1)
print('Total soil biomass from Stats (extended range): \n' + prnt_stats(soil_tots_SP))

samp_dist_tots_SP = cdf_2_tot(CDF_SP,area=biome_area1, N_samps = N_samps)
soil_Overall_tots_SP = samp_dist_tots_SP.sum(axis=0)
print("Total soil biomass from Monte-Carlo: \n"+ prnt_monte(soil_Overall_tots_SP))

plot_biomass(soil_Overall_tots_SP,color=Habitat_Color[0], xlabel = 'Dry Biomass in Soil [Mt]', ylabel='Probability Density\n[arb. units]',
             save_bool = save_bool , filename='Soil Biomass sub-phyla (including eff. mass)')

#tot_biomass_taxon_SP = samp_dist_tots_SP.groupby('aggregated taxon').sum().T

In [ ]:
unique_sites_per_biome = soil_site_data_mass_all_SP.groupby(['aggregated taxon','aggregated biome'])['site'].nunique().reset_index() #number of unique sites per biome
unique_sites_per_biome_SP = soil_site_data_mass_all_SP.groupby(['aggregated taxon','sub-phylum','aggregated biome'])['site'].nunique().reset_index() #number of unique sites per sub-phylum and biome
unique_sites = unique_sites_per_biome_SP.merge(unique_sites_per_biome,on=['aggregated taxon','aggregated biome'])
unique_sites.loc[:,'correction_term'] = unique_sites.site_x/unique_sites.site_y
unique_sites.drop(['site_x','site_y'],axis=1,inplace=True)
soil_tots_SP_correct = soil_tots_SP.merge(unique_sites,on=['aggregated taxon','sub-phylum','aggregated biome'])
soil_tots_SP_correct.loc[:,'Total_corrected'] = soil_tots_SP_correct.loc[:,'Total']*soil_tots_SP_correct.loc[:,'correction_term']

soil_tots_SP_correct.loc[:,'Total_corrected'] = soil_tots_SP_correct.loc[:,'Total']*soil_tots_SP_correct.loc[:,'correction_term']
SP_tot = soil_tots_SP_correct.groupby('sub-phylum').Total_corrected.sum().drop(['Mixed'])

In [ ]:
#aggregating soil data by: only biomes (extreme case with known biases)
b_Stats,b_tots, b_Overall_tots = calc_boot_full(data=soil_data_mass,GroupBy=['aggregated biome'],filename='biome only soil Biomass',
               N_boot=N_boot,N_samp=N_samps,Nbins=Nbins,
               save_bool=save_bool,xlabel = 'Dry Biomass in Soil [Mt]')

In [ ]:
#Sanity check for filtering\excluding cases with data on ants and termites where the sampling effort was not focused solely on them

sites_with_others = soil_site_data_mass_all[soil_site_data_mass_all['aggregated taxon']=='Other'].site.unique()

a = soil_site_data_mass_all[soil_site_data_mass_all['aggregated taxon'].isin(['Formicidae','Isoptera'])]
a = a[~a.site.isin(sites_with_others)]
b = a.append(soil_site_data_mass_all[~soil_site_data_mass_all['aggregated taxon'].isin(['Formicidae','Isoptera'])])

soil_Stats_b, CDF_b = boot(b,GroupBy=['aggregated taxon','aggregated biome'],N=N_boot)
soil_tots_b = dens_2_tot(soil_Stats_b,area=biome_area1)
print('Total soil biomass from Stats (extended range): \n' + prnt_stats(soil_tots_b))

samp_dist_tots = cdf_2_tot(CDF_b,area=biome_area1, N_samps = N_samps)
soil_Overall_tots_b = samp_dist_tots.sum(axis=0)
print("Total soil biomass from Monte-Carlo: \n"+ prnt_monte(soil_Overall_tots_b))

plot_biomass(soil_Overall_tots_b,color=Habitat_Color[0], xlabel = 'Dry Biomass in Soil [Mt]', ylabel='Probability Density\n[arb. units]',
             save_bool = False , filename='Soil Biomass')

In [ ]:
#Compare to the main estimate, with no such filtering
Totals_all = soil_tots.groupby(['aggregated taxon','aggregated biome']).Total.sum().reset_index().merge(soil_tots_b.groupby(['aggregated taxon','aggregated biome']).Total.sum().reset_index(),how='outer',on=['aggregated taxon','aggregated biome'])
Totals_all.rename(columns={'Total_x': 'Total_no_filt','Total_y': 'Total_filt'}, inplace=True)
Totals_all.groupby(['aggregated taxon']).sum()

In [ ]:
#exclude macroarthropods for Others
soil_data_excl_macro = soil_data[soil_data['studied group']!='Macroarthropods']

soil_site_data_mass_all_excl_macro = average_with_eff_mass(soil_data_excl_macro,GroupBy=['units type','aggregated taxon','aggregated biome']) #data to be used. Contains all soil data, converted into dry biomass
soil_Stats_excl_macro, CDF_excl_macro = boot(soil_site_data_mass_all_excl_macro,GroupBy=['aggregated taxon','aggregated biome'],N=N_boot)
soil_tots_excl_macro = dens_2_tot(soil_Stats_excl_macro,area=biome_area1)

Others_tots_comapre = soil_tots_excl_macro[soil_tots_excl_macro['aggregated taxon']=='Other'].loc[:,['aggregated biome','Total']].merge(soil_tots[soil_tots['aggregated taxon']=='Other'].loc[:,['aggregated biome','Total']],how='outer',on=['aggregated biome'])
Others_tots_comapre.rename(columns={'Total_x': 'Total_excluding_macrofauna','Total_y': 'Total_no_exclusion'}, inplace=True)
Others_tots_comapre

### Calculate the overall total population, assuming places with no data has no individuals

In [ ]:
#aggregating population in soil data by: taxon, biome 
soil_data_pop = soil_data[soil_data['units type']=='individuals']

soil_pop_Stats,soil_pop_tots, soil_pop_Overall_tots, soil_pop_dist = calc_boot_full(data=soil_data_pop,GroupBy=['aggregated taxon','aggregated biome'],filename='pop in soil',
               N_boot=N_boot,N_samp=N_samps,Nbins=Nbins,return_dist=True,
               is_population=True, save_bool=save_bool,xlabel = r'Soil Population [ind. $\times 10^{18}$]')

In [ ]:
above_ground_data_pop = above_ground_data[above_ground_data['units type']=='individuals']

above_ground_pop_Stats,above_ground_pop_tots, above_ground_pop_Overall_tots = calc_boot_full(data=above_ground_data_pop,GroupBy=['aggregated biome'],filename='pop above ground',
               N_boot=N_boot,N_samp=N_samps,Nbins=Nbins,color=Habitat_Color[2],
               is_population=True, save_bool=save_bool,xlabel = r'Above Ground Population [ind. $\times 10^{18}$]')

In [ ]:
#Total population:
Overall_pop = comb_tots(soil_pop_Overall_tots,above_ground_pop_Overall_tots,N_samps=N_samps,Nbins=Nbins)
print('Overall_population\n'+ prnt_monte_pop(Overall_pop))

Overall_low = soil_pop_tots.Total_low.sum()+above_ground_pop_tots.Total_low.sum()
Overall_high = soil_pop_tots.Total_high.sum()+above_ground_pop_tots.Total_high.sum()
print('Range: ' + prtF_pop(Overall_low) + ' - ' + prtF_pop(Overall_high) + ' individuals')

plot_biomass(Overall_pop,color='#1e2fc9', xlabel = r'Population [ind. $\times 10^{18}$]', ylabel='Probability Density\n[arb. units]',
             save_bool = save_bool , filename='All Population')

In [ ]:
soil_pop_tots.groupby('aggregated taxon')['Total','Total_low','Total_high'].sum()

In [ ]:
#Combine biome totals for plotting
Soil_mass = soil_tots.groupby('aggregated biome').sum().loc[:,['Total','Total_low','Total_high']].reset_index()
Soil_mass.insert(0, 'env', 'soil',True)

above_ground_mass = above_ground_tots.groupby('aggregated biome').sum().loc[:,['Total','Total_low','Total_high']].reset_index()
above_ground_mass.insert(0, 'env', 'above_ground',True)

biome_totals_mass = pd.concat([Soil_mass, above_ground_mass])

#population:
Soil_pop = soil_pop_tots.groupby('aggregated biome').sum().loc[:,['Total','Total_low','Total_high']].reset_index()
Soil_pop.insert(0, 'env', 'soil',True)

above_ground_pop = above_ground_pop_tots.groupby('aggregated biome').sum().loc[:,['Total','Total_low','Total_high']].reset_index()
above_ground_pop.insert(0, 'env', 'above_ground',True)

biome_totals_pop = pd.concat([Soil_pop, above_ground_pop])

In [ ]:
#Export to tables

def print_export_tot(mean,low,high,N=2,is_pop=False): #print the mean and range (low-high) rounded to N significant digits, for exporting data tables
    if is_pop:
        mean = round_significant_N(mean/1e15,N)
        low = round_significant_N(low/1e15,N)
        high = round_significant_N(high/1e15,N)
    else:
        mean = round_significant_N(mean,N)
        low = round_significant_N(low,N)
        high = round_significant_N(high,N)
    
    txt='{MEAN} ({LOW}-{HIGH})'
    return txt.format(MEAN=mean, LOW=low, HIGH=high)

def export_tot_table(biome_totals,is_pop,N): #convert the biomas totals DF to a pivotted table to be exported. is_pop=is the data a population data. N=number of significant digits
    biome_totals_exp = biome_totals.groupby(['env','aggregated biome']).mean() #groupby environment and biomes. The mean doesnt really do anything
    Habitat_totals_sums = biome_totals.groupby(['env']).sum()
    
    biome_totals_exp = biome_totals_exp.apply(lambda x : print_export_tot(x.Total, x.Total_low,x.Total_high,N=N,is_pop=is_pop),axis=1).reset_index()
    Habitat_totals_sums = Habitat_totals_sums.apply(lambda x : print_export_tot(x.Total, x.Total_low,x.Total_high,N=N,is_pop=is_pop),axis=1).reset_index()
    biome_totals_exp = pd.concat([biome_totals_exp,Habitat_totals_sums])
    biome_totals_exp.iloc[-len(Habitat_totals_sums):,1]='World Total' #rename the sum of all biomes (Habitat_totals_sums) to "World Total"

    biome_totals_table = pd.pivot(biome_totals_exp,index='aggregated biome', columns='env')
    return biome_totals_table
 
biome_totals_mass_table = export_tot_table(biome_totals_mass,is_pop=False,N=1)
biome_totals_pop_table = export_tot_table(biome_totals_pop,is_pop=True,N=1)

if save_bool:
    biome_totals_mass_table.to_csv('results/tables/table_biome_totals_mass.csv')
    biome_totals_pop_table.to_csv('results/tables/table_biome_totals_pop.csv')

### Calculate the total population per aggregated biome for later plotting

In [ ]:
def sort_by_index(x,sorted_indx):
    sorterIndex = dict(zip(sorted_indx, range(len(sorted_indx))))
    x['sorting_biomes'] = x.index.map(sorterIndex)
    x.sort_values(['sorting_biomes'],ascending = True, inplace = True)
    x.drop('sorting_biomes', axis=1, inplace = True)
    return x

def data4tot_plot(x):
    #rename labels for plot
    original_biomes = x['aggregated biome']
    transdict = {'Boreal Forests/Taiga': 'Boreal forests',
             'Crops':'Crops',
             'Deserts and Xeric Shrublands':'Deserts',
             'Mediterranean Forests, Woodlands and Scrub':'Mediterranean',
             'Pasture':'Pasture',
           'Temperate Forests':'Temp. forests',
           'Temperate Grasslands, Savannas and Shrublands':'Temp.\ngrasslands',
           'Tropical and Subtropical Forests':'Trop. forests',
           'Tropical and Subtropical Grasslands, Savannas and Shrublands':'Trop.\ngrasslands',
           'Tundra':'Tundra'
            }
    print_biomes = [transdict[biome] for biome in original_biomes]    
    x['aggregated biome'] = print_biomes
    
    
    #extract the total values
    x_tot = x.reset_index().pivot(index = 'aggregated biome' , columns = 'env', values='Total')    
    
    #sort by descending values
    sorter = x_tot.sum(axis=1).sort_values(ascending=False).index
    x_tot = sort_by_index(x_tot ,sorter )
    x_tot = x_tot.sort_index(axis=1,ascending=False) #make sure the soil column is first
    
    #add error bars
    x['err_low'] = (x.loc[:,'Total']-x.loc[:,'Total_low'])
    x['err_high'] = (x.loc[:,'Total_high']-x.loc[:,'Total'])
    
    err_l = x.pivot(index = 'aggregated biome' , columns = 'env', values='err_low' ) 
    err_l = sort_by_index(err_l,x_tot.index)
    err_l = err_l.sort_index(axis=1,ascending=False)
    err_h = x.pivot(index = 'aggregated biome' , columns = 'env', values= 'err_high')
    err_h = sort_by_index(err_l,x_tot.index)
    err_h = err_h.sort_index(axis=1,ascending=False)
    
    #Combine the low and high errors
    err = []
    for col in err_l:  # Iterate over bar groups (represented as columns)
        err.append([err_l[col].values, err_h[col].values])    
        
    return x_tot , err


mass_tots, err_m = data4tot_plot(biome_totals_mass)
pop_tots, err_p= data4tot_plot(biome_totals_pop)

## plot population density in soil

In [ ]:
plt.rcParams.update({'font.size': 10})
plt.rc('text', usetex=False)

#r_pop is the data to be plotted
r_pop = soil_site_data_pop.reset_index() #'population density' is the column being used later

#Sort according to biomes total mean biomass, in a descending order
sorter = soil_pop_Stats.reset_index().groupby('aggregated biome').sum().sort_values(by='mean',ascending=False).index
sorterIndex = dict(zip(sorter, range(len(sorter))))
r_pop['sorting_biomes'] = r_pop['aggregated biome'].map(sorterIndex)
r_pop.sort_values(['sorting_biomes'],ascending = True, inplace = True)
r_pop.drop('sorting_biomes', axis=1, inplace = True)

original_biomes = r_pop['aggregated biome'].unique()
transdict = {'Boreal Forests/Taiga': 'Boreal\nforests/\ntaiga\n(16M km$^2$)',
             'Crops':'Crops\n(15M km$^2$)',
             'Deserts and Xeric Shrublands':'Deserts\nand xeric\nshrublands\n(20M km$^2$)',
             'Mediterranean Forests, Woodlands and Scrub':'Mediterranean\nforests,\nwoodlands\nand scrub\n(1.6M km$^2$)',
             'Pasture':'Pasture\n(28M km$^2$)',
           'Temperate Forests':'Temperate\nforests\n(11M km$^2$)',
           'Temperate Grasslands, Savannas and Shrublands':'Temperate\ngrasslands,\nsavannas and\nshrublands\n(5.4M km$^2$)',
           'Tropical and Subtropical Forests':'Tropical and\nsubtropical\nforests\n(18M km$^2$)',
           'Tropical and Subtropical Grasslands, Savannas and Shrublands':'Tropical and\nsubtropical\ngrasslands,\nsavannas and\nshrublands\n(10M km$^2$)',
           'Tundra':'Tundra\n(11M km$^2$)'
            }

print_biomes = [transdict[biome] for biome in original_biomes]
r_pop.replace(to_replace=original_biomes,value=print_biomes,inplace=True)

bins = np.logspace(-1,np.log10(r_pop['population density'].max()),20)
bins = np.insert(bins,0,-0.1) #add the point -0.1 to the beginning of the bins array
  
t = r_pop.groupby(['aggregated biome','aggregated taxon'])['population density'].apply(lambda x: pd.cut(x,bins=bins).value_counts()/len(x)).reset_index() #how many in each bin
t_max = t.groupby(['aggregated biome','aggregated taxon'])['population density'].apply(np.max).reset_index() 
t_norm_pop = t.merge(t_max,left_on=['aggregated biome','aggregated taxon'],right_on=['aggregated biome','aggregated taxon']) #normalize to the maximal bin value
t_norm_pop['val'] = t_norm_pop['population density_x']/t_norm_pop['population density_y']

In [ ]:
def print_export_pop(mean,low,high,N_sites,N=2): #print the mean and range (low-high) rounded to N significant digits, for exporting data tables
    mean = round_significant_N(mean,N)
    low = round_significant_N(low,N)
    high = round_significant_N(high,N)
    
    if N_sites>1:
        txt='{MEAN}\n({LOW}-{HIGH})\nN = {N_SITES}'
    else: 
        txt='{MEAN}\n\nN = {N_SITES}'
    return txt.format(MEAN=mean, LOW=low, HIGH=high, N_SITES=int(N_sites))

soil_export_pop = soil_pop_Stats.reset_index().merge(soil_site_data_pop.groupby(['aggregated taxon','aggregated biome']).site.count().reset_index())
soil_export_pop = soil_export_pop.groupby(['aggregated taxon','aggregated biome']).mean()
soil_export_pop = soil_export_pop.apply(lambda x : print_export_pop(x['mean'], x['2.5%'],x['97.5%'],x['site'],N=1),axis=1).reset_index()
soil_export_pop.rename(columns={'aggregated taxon' : 'Aggregated Taxon', 'aggregated biome': 'Aggregated Biome',0: 'Density (95% Range)', 'site': '# Sites'}, inplace=True)
soil_density_table_pop = pd.pivot(soil_export_pop, values=['Density (95% Range)'], index='Aggregated Biome',
                    columns='Aggregated Taxon')

if save_bool:
    soil_density_table_pop.to_csv('results/tables/Soil population density table.csv')

# Additional sensitivity analysis

In [ ]:
#Quick check
N = len(soil_site_data_mass_all)
N_range = np.arange(0, N, 1)
SSd = soil_site_data_mass_all.loc[:,['aggregated taxon','aggregated biome','mass_G']]

tot_soil_sens=np.zeros(N)

for ii in N_range:
    soil_biome_means_G_sens = SSd.loc[N_range!=ii].groupby(['aggregated taxon','aggregated biome'])['mass_G'].agg(np.mean).reset_index()
                          
    soil_biome_Mean_sens = soil_biome_means_G_sens.merge(pd.DataFrame(biome_area1),left_on='aggregated biome',right_on='aggregated biome')
    tot_soil_sens[ii] = (soil_biome_Mean_sens.loc[:,'mass_G']*soil_biome_Mean_sens.loc[:,'area']).values.sum()/1e15 #total population per biome, in units of mg. (use 1e18 to convert to

plt.hist(tot_soil_sens,40)

#Removing a single site can result in up to 3% change in the final evaluation for soils
print('extreme upper change: ' + '{:.1f}'.format(100*(np.max(tot_soil_sens)-np.mean(tot_soil_sens))/np.mean(tot_soil_sens)) + '%')
print('extreme lower change: ' + '{:.1f}'.format(100*(np.mean(tot_soil_sens)-np.min(tot_soil_sens))/np.mean(tot_soil_sens)) + '%')

In [ ]:
def filter_outliers(x):
    if len(x)>3:
        STD=x.std()
        Mean=np.mean(x)
        x = x[x<(Mean+2*STD)]
        x = x[x>(Mean-2*STD)]
    return x

def filter_outliers_log(x):
    if len(x)>3:
        X = np.log(x+0.01)
        STD=X.std()
        Mean=np.mean(X)
        x = x[X<(Mean+2*STD)]
        x = x[X>(Mean-2*STD)]
    return x

def filt_mean(x):
    return np.mean(filter_outliers_log(x))

soil_biome_means_G_outliers = SSd.groupby(['aggregated taxon','aggregated biome'])['mass_G'].agg([filt_mean]).reset_index()
soil_biome_Mean_outliers = soil_biome_means_G_outliers.merge(pd.DataFrame(biome_area1),left_on='aggregated biome',right_on='aggregated biome')
tot_soil_outliers = (soil_biome_Mean_outliers.loc[:,'filt_mean']*soil_biome_Mean_outliers.loc[:,'area']).sum()/1e15 #total population per biome, in units of mg.

tot_soil_outliers
    

In [ ]:
#% change of removing all outliers
(soil_tots.Total.sum() - tot_soil_outliers)/soil_tots.Total.sum()

In [ ]:
#Calculate the percent difference between converting population to biomass on biome-level (_B) or global level (_G)
soil_means = soil_site_data_mass_all.groupby(['aggregated taxon','aggregated biome'])['mass_G','mass_B'].agg(np.mean)
tot_soil = soil_means.groupby(['aggregated biome']).sum().merge(pd.DataFrame(biome_area1),left_on='aggregated biome',right_on='aggregated biome')

print('The difference in total biomass between using global and biome-level convertions (from population to biomass) is:\n' +
    '%d' %((np.sum(tot_soil.mass_G*tot_soil.area)/np.sum(tot_soil.mass_B*tot_soil.area)-1)*100) +'%')

In [ ]:
tot_soil_mass = soil_site_data_mass_all.groupby(['aggregated taxon','aggregated biome'])['mass density'].agg(np.mean).groupby(['aggregated biome']).sum().reset_index().merge(pd.DataFrame(biome_area1),left_on='aggregated biome',right_on='aggregated biome')
tot_soil_mass['mass density']/tot_soil.mass_G

In [ ]:
soil_site_Other = soil_site_data_mass_all[soil_site_data_mass_all['aggregated taxon'] == 'Other']
SP_soil_site_Other = soil_site_data_mass_all_SP[soil_site_data_mass_all_SP['aggregated taxon'] == 'Other']

In [ ]:
SP_mean1 = soil_site_Other.groupby(['aggregated biome'])['mass_G'].mean()
SP_mean2 = SP_soil_site_Other.groupby(['sub-phylum','aggregated biome'])['mass_G'].mean().reset_index().groupby(['aggregated biome'])['mass_G'].sum()
SP_mean2/SP_mean1

## Figures for paper

### Figure 2 (soil densities)

In [ ]:
def print_data(x,ax):
    col_map = pd.Series(col_arr,index=['Acari','Collembola','Formicidae','Isoptera','Other'])
    fine_loc = pd.Series([-0.4,-0.2,0,0.2,0.4],index=['Acari','Collembola','Formicidae','Isoptera','Other'])
    col = col_map.loc[x['aggregated taxon'].values[0]]
    locs = pd.Series(range(0,2*len(r['aggregated biome'].unique()),2),index=r['aggregated biome'].unique())
    xloc = locs.loc[x['aggregated biome'].values[0]] + fine_loc.loc[x['aggregated taxon'].values[0]]
    if x['val'].values[0] == 1:
        lab = x['aggregated taxon'].values[0]
    else:
        lab=None
    plt.bar(x=xloc,
            height=x['level_2'].values[0].right-x['level_2'].values[0].left,
            width=0.17,
            bottom=x['level_2'].values[0].left,
            log=True,color=col,alpha=x['val'].values[0],
            label=lab,
            zorder=1
           )
    ax.set_yscale('symlog',linthreshy=0.1)
        
def print_bootstrap(x):
    locs = pd.Series(range(0,2*len(r['aggregated biome'].unique()),2),index=r['aggregated biome'].unique())
    x = x.replace(to_replace=original_biomes,value=print_biomes)
    xloc = locs.loc[x['aggregated biome'].values[0]] + fine_loc.loc[x['aggregated taxon'].values[0]]
    Y = x['mean']+0.08
    L = x['2.5%']
    U = x['97.5%']
    plt.scatter(x=xloc,y= Y,marker='o' ,facecolors='w',edgecolors='k', alpha=1,s=21,linewidths=1.5,zorder=3)

fig,ax=plt.subplots(2,1,figsize=[12,11])

fine_loc = pd.Series([-0.4,-0.2,0,0.2,0.4],index=['Acari','Collembola','Formicidae','Isoptera','Other'])

ax1 = plt.subplot(2,1,1)

for i in range(0,2*len(r['aggregated biome'].unique()),2):    
    plt.plot([i-0.7,i+0.7],[0.05,0.05],'k')   #plot lines for labels
    plt.plot([i-0.7,i+0.7],[1e-1,1e-1],'k:',alpha = 0.8)    #plot dotted lines for breaking points "detection limit"

#Plot biomass data
t_norm.groupby(['aggregated biome','aggregated taxon','level_2']).apply(lambda x: print_data(x,ax=ax1)) #plot bins
#ax1 = plt.gca()
ax1.plot([-1,19],[10,10],'k-.',alpha=0.3,linewidth=0.5,zorder=0) #plot grid lines
ax1.plot([-1,19],[1000,1000],'k-.',alpha=0.3,linewidth=0.5,zorder=0) #plot grid lines
ax1.set_xticks(range(0,2*len(r['aggregated biome'].unique()),2)) 
ax1.set_xticklabels(r['aggregated biome'].unique(),fontdict={'size':10})
ax1.get_label()
ax1.set_ylabel('Dry biomass density in soil [mg/m$^2$]',fontdict={'size':14})
ax1.set_ylim([0.05,0.8e5])
ax1.set_xlim([-1,19])
legend_without_duplicate_labels(ax1,(0.5,1.1)) 
sb.despine(top=True, bottom=True, right=True)

soil_Stats.reset_index().groupby(['aggregated biome','aggregated taxon']).apply(print_bootstrap)

ax1.text(0.007, 0.123, 'Detection limit', transform=ax1.transAxes, fontsize=9,
        verticalalignment='top', bbox=None, color = 'k', alpha = 0.95)

ax2 = plt.subplot(2,1,2)

for i in range(0,2*len(r_pop['aggregated biome'].unique()),2):    
    plt.plot([i-0.7,i+0.7],[0.05,0.05],'k')   #plot lines for labels
    plt.plot([i-0.7,i+0.7],[1e-1,1e-1],'k:',alpha = 0.8)    #plot dotted lines for breaking points "detection limit"

#Plot population data
t_norm_pop.groupby(['aggregated biome','aggregated taxon','level_2']).apply(lambda x: print_data(x,ax=ax2))
#ax2 = plt.gca()
ax2.plot([-1,19],[1e1,1e1],'k-.',alpha=0.3,linewidth=0.5,zorder=0)
ax2.plot([-1,19],[1e3,1e3],'k-.',alpha=0.3,linewidth=0.5,zorder=0)
ax2.plot([-1,19],[1e5,1e5],'k-.',alpha=0.3,linewidth=0.5,zorder=0)
ax2.set_xticks(range(0,2*len(r['aggregated biome'].unique()),2))
ax2.set_xticklabels(r['aggregated biome'].unique(),fontdict={'size':10})
ax2.get_label()
ax2.set_ylabel('Population density in soil [ind./m$^2$]',fontdict={'size':14})
ax2.set_ylim([0.05,5e6])
ax2.set_xlim([-1,19])
#legend_without_duplicate_labels(ax2,(0.5,1.03)) #lower left 0.007 (0.01,0.9)
sb.despine(top=True, bottom=True, right=True)
soil_pop_Stats.reset_index().groupby(['aggregated biome','aggregated taxon']).apply(print_bootstrap)

ax2.text(0.007, 0.105, 'Detection limit', transform=ax2.transAxes, fontsize=9,
        verticalalignment='top', bbox=None, color = 'k', alpha = 0.95)

plt.tight_layout()

if save_bool:
    plt.savefig('results/figs/Soil densities (fig2).pdf',dpi=300)
else:
    plt.show()

### Figure 5S - biomass and population global distributions and total by biome - Soil + Above-ground

In [ ]:
fig,ax = plt.subplots(3,2,figsize=[9,9],dpi=300)
plt.rc('xtick', labelsize=12) 
plt.rc('ytick', labelsize=12)

plt.rcParams.update({'font.size': 8}) #change the font size here

xlabel_size = 12 #xlabel font size
ylabel_size = 12 #ylabel font size
letter_size = 16 #letter font size

def add_letter(ax,letter,pos_x = 0.01,pos_y = 0.89,letter_size = letter_size):
    ax.text(pos_x,pos_y,letter,fontdict={'size':letter_size,'weight':'bold'},transform = ax.transAxes)

def plot_scatter_on_bars(ax,colors):
    L=len(ax.patches)
    Li2 = int(len(ax5.patches)/2)
    ii=0
    x_b = np.empty(L)
    y_b = np.empty(L)

    for bar in ax.patches:    
        x_b[ii], y_b[ii] = (bar.get_x() + bar.get_width() / 2, bar.get_height())
        ii+=1

    plt.scatter(x_b[:Li2],y_b[:Li2],color=colors[0])
    plt.scatter(x_b[Li2:],y_b[Li2:],color=colors[1])

    
ax1 = plt.subplot(3,2,1)

plt.hist(soil_Overall_tots,bins=200,alpha=1, color=Habitat_Color[0])
plt.hist(above_ground_Overall_tots,bins=200,alpha=1, color=Habitat_Color[2])

plt.xlabel('Global dry biomass [Mt]',size = xlabel_size)
plt.ylabel('Probability density\n[arb. units]',size = ylabel_size)

ax1.xaxis.set_ticks([0,100,200,300])
ax1.xaxis.set_minor_locator(MultipleLocator(50))
mx=ax1.get_ylim()[1]*0.95 
ax1.yaxis.set_ticks([0, mx/2, mx])
ax1.yaxis.set_ticklabels([0, 0.5, 1])
add_letter(ax1,'A')

plt.legend(['Soil','Above ground'],frameon=False,loc=[0.01,1.02],ncol=2,fontsize=12)


ax2 = plt.subplot(3,2,2)

plt.hist(soil_pop_Overall_tots*1e18,bins=200,alpha=1, color=Habitat_Color[0])
plt.hist(above_ground_pop_Overall_tots*1e18,bins=200,alpha=1, color=Habitat_Color[2])

plt.xscale('log')
plt.xlabel('Global population [ind.]',size = xlabel_size)
plt.ylabel('Probability density\n[arb. units]',size = ylabel_size)


mx=ax2.get_ylim()[1]*0.95
ax2.yaxis.set_ticks([0, mx/2, mx])
ax2.yaxis.set_ticklabels([0, 0.5, 1])
add_letter(ax2,'B')

#Plot biomass and population data per biomeand habitat type (environment)

ax3 = plt.subplot(3,2,3)
mass_tots.plot.bar(width = 0.8, color=col_env,ax=ax3,logy=False,stacked=True,legend=False,use_index=False)
ax3.set_ylabel('Biome dry biomass [Mt]',size = ylabel_size)
ax3.xaxis.set_ticklabels(['','','','','','','','','',''])
ax3.set_ylim([0,103])
add_letter(ax3,'C')

ax4 = plt.subplot(3,2,4)
(pop_tots/1e18).plot.bar(width = 0.8, color=col_env,ax=ax4,logy=False,stacked=True,legend=False,use_index=False)
ax4.set_ylabel('Biome population\n' + r'[ind. $\times 10^{18}$]',size = ylabel_size)
ax4.xaxis.set_ticklabels(['','','','','','','','','',''])
ax4.set_ylim([0,4.1])
add_letter(ax4,'D')

ax5 = plt.subplot(3,2,5)
mass_tots.plot.bar(yerr=err_m[0],error_kw=dict(ecolor='black', lw=1, capsize=4, capthick=1) ,width = 0.8, color='None',ax=ax5,logy=True,stacked=False,rot=90,legend=False)
plot_scatter_on_bars(ax5,col_env)
ax5.set_ylabel('Biome dry biomass [Mt]',size = ylabel_size)
ax5.xaxis.label.set_visible(False)
ax5.set_ylim([0.11,230])
add_letter(ax5,'E')

ax6 = plt.subplot(3,2,6)
pop_tots.plot.bar(yerr=err_p[0],error_kw=dict(ecolor='black', lw=1, capsize=4, capthick=1) ,width = 0.8, color='None',ax=ax6,logy=True,stacked=False,rot=90,legend=False)
plot_scatter_on_bars(ax6,col_env)
ax6.set_ylabel('Biome population [ind.]',size = ylabel_size)
ax6.xaxis.label.set_visible(False)
ax6.set_ylim([1.1e15,3e19])
add_letter(ax6,'F')

plt.tight_layout()

if save_bool:
    plt.savefig('results/figs/Global and biome level total estimates (figS5).pdf',dpi=300,transparent=True)
else:
    plt.show()
    
plt.rcParams.update({'font.size': 10}) #change back the font size here

### Figure 3 - Main text (biomass and population global distributions and total by biome)

In [ ]:
mass_tots_s, err_m_s = data4tot_plot(Soil_mass)
pop_tots_s, err_p_s= data4tot_plot(Soil_pop)

In [ ]:
fig,ax = plt.subplots(2,2,figsize=[9,6],dpi=300)
plt.rc('xtick', labelsize=12) 
plt.rc('ytick', labelsize=13)

xlabel_size = 13 #xlabel font size
ylabel_size = 13 #ylabel font size
letter_size = 16 #letter font size

def add_letter(ax,letter,pos_x = 0.01,pos_y = 0.89,letter_size = letter_size):
    ax.text(pos_x,pos_y,letter,fontdict={'size':letter_size,'weight':'bold'},transform = ax.transAxes)

#Plot biomass and population data per biomeand habitat type (environment)

ax3 = plt.subplot(2,2,1)
mass_tots_s.plot.bar(y = 'soil', width = 0.8, color=col_soil,ax=ax3,logy=False,stacked=True,legend=False,use_index=False)
ax3.set_ylabel('Biome dry biomass [Mt]',size = ylabel_size)
ax3.xaxis.set_ticklabels(['','','','','','','','','',''])
ax3.set_ylim([0,65])
add_letter(ax3,'A')

ax4 = plt.subplot(2,2,2)
(pop_tots_s/1e18).plot.bar(y = 'soil',width = 0.8, color=col_soil,ax=ax4,logy=False,stacked=True,legend=False,use_index=False)
ax4.set_ylabel('Biome population\n' + r'[ind. $\times 10^{18}$]',size = ylabel_size)
ax4.xaxis.set_ticklabels(['','','','','','','','','',''])
ax4.set_ylim([0,4.1])
add_letter(ax4,'B')


ax5 = plt.subplot(2,2,3)
mass_tots_s.plot(y = 'soil', ax=ax5, marker='.',markersize=10,linestyle = 'None',color=col_soil,legend=False)
mass_tots_s.plot.bar(y = 'soil',yerr=err_m_s,error_kw=dict(ecolor='black', lw=1, capsize=4, capthick=1) ,width = 0.8, color='white',ax=ax5,logy=True,stacked=False,rot=90,legend=False)
ax5.set_ylabel('Biome dry biomass [Mt]',size = ylabel_size)
ax5.xaxis.label.set_visible(False)
ax5.set_ylim([0.11,230])
add_letter(ax5,'C')

ax6 = plt.subplot(2,2,4)
pop_tots_s.plot(y = 'soil', ax=ax6,marker='.',markersize=10,linestyle = 'None',color=col_soil,legend=False)
pop_tots_s.plot.bar(y = 'soil',yerr=err_p_s,error_kw=dict(ecolor='black', lw=1, capsize=4, capthick=1) ,width = 0.8, color='white',ax=ax6,logy=True,stacked=False,rot=90,legend=False)
ax6.set_ylabel('Biome population [ind.]',size = ylabel_size)
ax6.xaxis.label.set_visible(False)
ax6.set_ylim([1.1e15,3e19])
add_letter(ax6,'D')

plt.tight_layout()

if save_bool:
    plt.savefig('results/figs/fig3 - Global and biome level total estimates - soil.pdf',dpi=300,transparent=True)
else:
    plt.show()

### Figure 4 (soil totals by taxon)

In [ ]:
fig,ax = plt.subplots(1,2,figsize=[9,3],dpi=300)
plt.rc('xtick', labelsize=13) 
plt.rc('ytick', labelsize=13)

xlabel_size = 13 #xlabel font size
ylabel_size = 13 #ylabel font size

#plt.rcParams.update({'font.size': 12}) #increase the font size here
plt.rc('text', usetex=False)



#plt.rcParams.update({'font.size': 10}) #decrease back the font size here




#Biomass bars by taxon
ax1 = plt.subplot(1,2,1)

group_sum = tot_biomass_taxon.median()
group_colors = pd.Series(col_arr,index = group_sum.index)
new_index = ['Isoptera','Acari','Formicidae','Collembola','Other']
group_sum = group_sum.reindex(new_index)
group_colors = group_colors.reindex(new_index)

group_sum.index = ['Isoptera\n(termites)','Acari\n(mites)','Formicidae\n(ants)','Collembola\n(springtails)','Others']

group_colors.index = group_sum.index
group_sum.plot.bar(width = 0.6,color=group_colors.loc[group_sum.index],rot=0,ax=ax1)
ax1.set_ylabel('Dry biomass in soil [Mt]',fontdict={'size':ylabel_size})
plt.xticks(fontsize=8)

ax1.set_ylim([0,140])
ax1.set(yticks=np.arange(0,140,50))

add_letter(ax1,'A',pos_x = 0.02,pos_y = 0.9)

#Biomass percentage by sub-phyla
ax2 = plt.subplot(1,2,2)

subphyla_sum = SP_tot.sort_values(ascending=False)
labels = subphyla_sum.index + '     '


patches, texts, pcts = ax2.pie(
    subphyla_sum, labels=labels, 
    colors=['#65330A', '#C66300','#ADA887','#3C463B'],
    autopct='%1.0f%%',pctdistance=0.84,textprops={'color':"k"},
    wedgeprops={'width':0.33,'edgecolor':'black', 'linewidth':0.5},
    startangle=180,explode = (0.01, 0.05, 0.05, 0.05))

# Style just the percent values.
plt.setp(pcts, color='white', fontweight='bold')

add_letter(ax2,'B',pos_x = -0.45,pos_y = 0.9)

plt.tight_layout()

if save_bool:
    plt.savefig('results/figs/Soil taxonomic breakdown (fig4).pdf',dpi=300)
else:
    plt.show()

### Figure 5 - Global total biomass with supp. estimate for above-ground arthropods

In [ ]:
plt.rc('xtick', labelsize=18) 
plt.rc('ytick', labelsize=18)

xlabel_size = 18 #xlabel font size
ylabel_size = 18 #ylabel font size

#plt.rcParams.update({'font.size': 12}) #increase the font size here
plt.rc('text', usetex=False)



totals = [['Soil and litter', 220,104,378], ['Above-ground', 54,22,106]] #106
tots = pd.DataFrame(totals, columns=['habitat', 'Mean','Min','Max'])

errors = np.array([tots.Mean-tots.Min,tots.Max-tots.Mean])

tots.plot(kind = "bar",x='habitat', y = "Mean", legend = False, title = "", rot=0,
          yerr = errors, color=col_env, alpha=0.9,error_kw=dict(ecolor='black', lw=3, capsize=0, capthick=3))
plt.ylabel('Global dry biomass [Mt]',size = ylabel_size)

plt.tight_layout()
#plt.show()
if True:
    plt.savefig('results/figs/total_biomass_fig5.pdf',dpi=300)
#    plt.savefig('results/figs/png/total_biomass_fig5.png',dpi=300)

### Export above-ground max densities and techniques used

In [ ]:
#Maximal densities for each biome
X = averaged_sum_site(above_ground_data_mass,GroupBy=['aggregated biome'])

X.loc[X['standard deviation'] == 0,'standard deviation'] =  X.loc[X['standard deviation'] == 0,'norm value']/2
X.loc[:,'p97'] = X.loc[:,'norm value']+ 2*X.loc[:,'standard deviation']
X.loc[:,'Max'] = X.iloc[:,2:].max(axis=1)
X.groupby('aggregated biome').Max.max().round()

In [ ]:
X_max = X.groupby('aggregated biome').Max.max().reset_index().merge(biome_area1)
X_max.loc[:,'Total'] = X_max.loc[:,'Max'] * X_max.loc[:,'area']
np.round(X_max.Total.sum()/1e15)

In [ ]:
if save_bool:
    X_max.set_index('aggregated biome').loc[:,'Max'].round().to_csv('results/tables/max_densities.csv')

In [ ]:
#extract techniques used for above-ground data
X_above = above_ground_data_mass.drop_duplicates(subset='site')
X_above.loc[:,'Tech'] = 'Trap and remove'
X_above.loc[(X_above['sampling technique'] == 'fogging','Tech')] = 'Canopy fogging'
X_above.loc[(X_above['sampling technique'] == 'D-vac','Tech')] = 'Vacuume suction'
X_above.loc[(X_above['sampling technique'] == 'hand capture + sweep nets','Tech')]  = 'Hand capture + Sweep nets'

Tech_table = X_above.pivot_table(index='aggregated biome',columns='Tech',values='site', aggfunc=['count'])

if save_bool:
    Tech_table.to_csv('results/tables/techniques_biomass.csv')